In [5]:
import pandas as pd
from pyproj import Proj, transform
from geopy.geocoders import Nominatim
import pyproj
import time
import re
import numpy as np
from tqdm import tqdm
from geopy.exc import GeocoderTimedOut
import math

In [7]:
und = pd.read_csv("Logradouro tratado.csv", encoding='latin1', delimiter=';')
log = pd.read_csv("Unidade comercial tratado.csv", encoding='latin1', delimiter=';')
resultado = pd.concat([und, log]).reset_index(drop=True)
resultado

,Unnamed: 0,Matrícula,Categoria de serviço,Data de Solicitação,Endereço,Bairro,Número,Latitude,Longitude
0,2,NaN,Vazamento,02/01/2023 08:36:00,Rua Mal. Castelo Branco,Bracinho,NaN,NaN,NaN
1,7,NaN,Vazamento,06/01/2023 08:19:00,Rua Barao Do Rio Branco,Centro Leste,NaN,NaN,NaN
2,16,NaN,Vazamento,10/01/2023 10:08:00,Rua Guilherme Piske,Centro Norte,NaN,NaN,NaN
3,17,NaN,Vazamento,10/01/2023 14:36:00,Rua Henrique Ziebel,Rio Hern,NaN,NaN,NaN
4,25,NaN,Vazamento,16/01/2023 13:33:00,Rua Jorge Lacerda,Centro Norte,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
69,13484,1365896-4,Falta de água,25/05/2023 15:46:00,Rua 573- VERONICA WELTER 156 - Schroeder I -,Schroeder I,NaN,NaN,NaN
70,17282,1369024-8,Falta de água,19/06/2023 15:10:00,R. Santa. Catarina 15 - Braco do Sul -,Braco do Sul,NaN,NaN,NaN
71,20006,801221-0,Falta de água,30/06/2023 10:13:00,R. Duque de Caxias 381 - Centro Norte -,Centro Norte,NaN,NaN,NaN
72,29384,1354221-4,Falta de água,08/09/2023 14:21:00,R. PRES. COSTA E SILVA 1291 - Rio Hern -,Rio Hern,NaN,NaN,NaN


In [8]:
resultado['Endereço'] = resultado['Endereço'].str.rstrip()
resultado['Endereço'] = resultado['Endereço'].apply(lambda x: x.rstrip('-') if x.endswith('-') else x)
resultado['Endereço'] = resultado['Endereço'].str.replace('Serv.', 'Servidão')
resultado['Endereço'] = resultado['Endereço'].str.replace('R.', 'Rua')
resultado['Endereço'] = resultado['Endereço'].apply(lambda x: re.sub(r'\([^()]*\)', '', x))
resultado['Endereço'] = resultado['Endereço'].str.title()
resultado

,Unnamed: 0,Matrícula,Categoria de serviço,Data de Solicitação,Endereço,Bairro,Número,Latitude,Longitude
0,2,NaN,Vazamento,02/01/2023 08:36:00,Rua Mal. Castelo Branco,Bracinho,NaN,NaN,NaN
1,7,NaN,Vazamento,06/01/2023 08:19:00,Rua Barao Do Rio Branco,Centro Leste,NaN,NaN,NaN
2,16,NaN,Vazamento,10/01/2023 10:08:00,Rua Guilherme Piske,Centro Norte,NaN,NaN,NaN
3,17,NaN,Vazamento,10/01/2023 14:36:00,Rua Henrique Ziebel,Rio Hern,NaN,NaN,NaN
4,25,NaN,Vazamento,16/01/2023 13:33:00,Rua Jorge Lacerda,Centro Norte,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
69,13484,1365896-4,Falta de água,25/05/2023 15:46:00,Rua 573- Veronica Welter 156 - Schroeder I,Schroeder I,NaN,NaN,NaN
70,17282,1369024-8,Falta de água,19/06/2023 15:10:00,Rua Santa. Catarina 15 - Braco Do Sul,Braco do Sul,NaN,NaN,NaN
71,20006,801221-0,Falta de água,30/06/2023 10:13:00,Rua Duque De Caxias 381 - Centro Norte,Centro Norte,NaN,NaN,NaN
72,29384,1354221-4,Falta de água,08/09/2023 14:21:00,Rua Pres. Costa E Silva 1291 - Rio Hern,Rio Hern,NaN,NaN,NaN


In [11]:
cidade = 'Schroeder'
estado = 'Santa Catarina'
base = resultado

geolocator = Nominatim(user_agent="myGeocoder")
coordenadas = []


# Iterando pelas linhas do DataFrame e obtendo as coordenadas
for index, row in tqdm(base.iterrows(), total=len(base), desc="Geocodificação"):
    rua = row['Endereço']

    # Primeiro, tenta obter as coordenadas com a rua e número
    if not pd.isnull(rua):
        endereco = f"{rua} - {cidade} - {estado}"
    location = None

    try:
        location = geolocator.geocode(endereco)
        #print(location)

        if location:
            coordenadas.append((location.latitude, location.longitude,location))
        else:
            coordenadas.append((np.nan, np.nan, np.nan))
            
    except Exception as e:
        coordenadas.append((np.nan, np.nan, np.nan))

    time.sleep(1)  # Adiciona um pequeno intervalo para evitar bloqueios por uso excessivo da API


# Criando as colunas 'x', 'y', 'Bairro' e 'Cidade' no DataFrame e atribuindo as coordenadas
resultado['x'] = [coord[0] for coord in coordenadas]
resultado['y'] = [coord[1] for coord in coordenadas]
resultado['Localização'] = [coord[2] for coord in coordenadas]

# Criando um objeto transformer para a transformação das coordenadas
transformer = pyproj.Transformer.from_crs("epsg:4326", "epsg:32722", always_xy=True)

# Aplicando a transformação das coordenadas e atribuindo os resultados a 'Latitude' e 'Longitude'.
resultado['Longitude'], resultado['Latitude'] = zip(*resultado.apply(lambda row: transformer.transform(row['y'], row['x']), axis=1))

# Exibindo o DataFrame com as coordenadas transformadas
display(resultado)

Geocodificação: 100%|██████████████████████████████████████████████████████████████████| 74/74 [01:40<00:00,  1.36s/it]


,Unnamed: 0,Matrícula,Categoria de serviço,Data de Solicitação,Endereço,Bairro,Número,Latitude,Longitude,x,y,Localização
0,2,NaN,Vazamento,02/01/2023 08:36:00,Rua Mal. Castelo Branco,Bracinho,NaN,7.074743e+06,693005.217413,-26.434517,-49.064493,"(Rua Marechal Castelo Branco, Centro Sul, Schr..."
1,7,NaN,Vazamento,06/01/2023 08:19:00,Rua Barao Do Rio Branco,Centro Leste,NaN,7.079385e+06,692995.054867,-26.392632,-49.065294,"(Rua Barão do Rio Branco, Centro Leste, Schroe..."
2,16,NaN,Vazamento,10/01/2023 10:08:00,Rua Guilherme Piske,Centro Norte,NaN,7.078325e+06,691093.114741,-26.402450,-49.084194,"(Rua Guilherme Piske, Centro Norte, Schroeder,..."
3,17,NaN,Vazamento,10/01/2023 14:36:00,Rua Henrique Ziebel,Rio Hern,NaN,7.075769e+06,693307.762184,-26.425218,-49.061615,"(Rua Henrique Ziebel, Rio Hern, Schroeder, Reg..."
4,25,NaN,Vazamento,16/01/2023 13:33:00,Rua Jorge Lacerda,Centro Norte,NaN,7.079241e+06,691187.500558,-26.394173,-49.083384,"(Rua Jorge Lacerda, Sossego, Schroeder, Região..."
...,...,...,...,...,...,...,...,...,...,...,...,...
69,13484,1365896-4,Falta de água,25/05/2023 15:46:00,Rua 573- Veronica Welter 156 - Schroeder I,Schroeder I,NaN,NaN,NaN,NaN,NaN,NaN
70,17282,1369024-8,Falta de água,19/06/2023 15:10:00,Rua Santa. Catarina 15 - Braco Do Sul,Braco do Sul,NaN,7.079338e+06,691265.804666,-26.393289,-49.082614,"(Rua Santa Catarina, Sossego, Schroeder, Regiã..."
71,20006,801221-0,Falta de água,30/06/2023 10:13:00,Rua Duque De Caxias 381 - Centro Norte,Centro Norte,NaN,7.078714e+06,690998.662427,-26.398955,-49.085198,"(Rua Duque de Caxias, Centro Norte, Schroeder,..."
72,29384,1354221-4,Falta de água,08/09/2023 14:21:00,Rua Pres. Costa E Silva 1291 - Rio Hern,Rio Hern,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
resultado.to_excel("Resultado.xlsx")

In [15]:
# Filtrar dados para a categoria de serviço 'Vazamento'
vazamento_data = resultado[resultado['Categoria de serviço'] == 'Falta de água']

# Agrupar por 'Bairro' e contar as ocorrências
contagem_por_bairro = vazamento_data['Bairro'].value_counts()

# Exibir o resultado
print(contagem_por_bairro)


Bairro
Rio Hern          7
Schroeder  I .    3
Centro            3
Duas Mamas        1
Itoupava Acu      1
Centro Sul        1
Schroeder I       1
Braco do Sul      1
Centro Norte      1
Name: count, dtype: int64
